In [2]:
import json
import random
import os

# Define the number of samples to generate
num_samples = 6

# Possible gait types
gaits = ["wave", "tripod", "ripple"]

# Directory for images
image_dir = "images"

# Ensure the directory exists
os.makedirs(image_dir, exist_ok=True)

# Metadata list
metadata = {}

for i in range(1, num_samples + 1):
    filename = f"images/{str(i).zfill(3)}.png"

    # Generate random movement and position values
    data = {
        "Body Movement": {
            "vx": round(random.uniform(0.1, 1.0), 2),
            "vy": round(random.uniform(-0.5, 0.5), 2),
            "v_rot": round(random.uniform(-0.5, 0.5), 2)
        },
        "Movement parameters": {
            "step_height": round(random.uniform(0.02, 0.1), 2),
            "step_duration": round(random.uniform(1.0, 2.0), 2),
            "gait_select": random.choice(gaits)
        },
        "Body Position": {
            "pos_x": round(random.uniform(-0.2, 0.2), 2),
            "pos_y": round(random.uniform(-0.2, 0.2), 2),
            "pos_z": round(random.uniform(0.05, 0.2), 2)
        },
        "Body Rotation": {
            "roll": round(random.uniform(-10, 10), 2),
            "pitch": round(random.uniform(-10, 10), 2),
            "yaw": round(random.uniform(-180, 180), 2)
        }
    }

    # Store metadata in dictionary
    metadata[filename] = data

# Save metadata as JSON
with open("metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Generated {num_samples} samples and saved metadata.json")


Generated 6 samples and saved metadata.json


In [9]:
from datasets import load_dataset
import json
import os

# Load dataset
dataset = load_dataset("imagefolder", data_dir="datasets", split="train")

# Load metadata
metadata_path = "datasets/metadata.json"
if os.path.exists(metadata_path):
    with open(metadata_path, "r") as f:
        metadata = json.load(f)
else:
    raise FileNotFoundError("metadata.json not found!")

# Print metadata keys (to check if file names match)
print("Metadata keys:", list(metadata.keys())[:5])  # Print first 5 keys for debugging

# Function to add text descriptions
def add_text(example):
    # Convert filename path format (fixes Windows/Linux path differences)
    file_name = os.path.basename(example["image"].filename)  # Extract "001.png"
    file_path = f"images/{file_name}"  # Match JSON key format

    # Debugging: Print file name and expected key
    print(f"Looking for: {file_path}")

    # Assign text from metadata, default to "No description available"
    example["text"] = metadata.get(file_path, "No description available")
    return example

# Apply function
dataset = dataset.map(add_text)

# Print results
print(dataset)
for i in range(6):
    print(dataset[i])  # Should now show {'image': ..., 'text': "Robot moves forward with wave gait."}


Metadata keys: ['images/001.png', 'images/002.png', 'images/003.png', 'images/004.png', 'images/005.png']
Dataset({
    features: ['image', 'text'],
    num_rows: 6
})
{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=251x499 at 0x14E1434A580>, 'text': {'Body Movement': {'v_rot': 0.32, 'vx': 0.2, 'vy': -0.04}, 'Body Position': {'pos_x': 0.06, 'pos_y': -0.09, 'pos_z': 0.17}, 'Body Rotation': {'pitch': -0.69, 'roll': -6.2, 'yaw': 165.6}, 'Movement parameters': {'gait_select': 'tripod', 'step_duration': 1.49, 'step_height': 0.03}}}
{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=347x689 at 0x14E1434A970>, 'text': {'Body Movement': {'v_rot': -0.23, 'vx': 0.83, 'vy': -0.07}, 'Body Position': {'pos_x': 0.03, 'pos_y': 0.14, 'pos_z': 0.06}, 'Body Rotation': {'pitch': 9.73, 'roll': -2.44, 'yaw': 42.82}, 'Movement parameters': {'gait_select': 'tripod', 'step_duration': 1.32, 'step_height': 0.03}}}
{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=287x4

In [11]:
import json
import os
from datasets import Dataset, Image
from PIL import Image as PILImage

# Load JSON dataset
with open("datasets/metadata.json", "r") as f:
    data = json.load(f)

# Convert to list format
dataset_list = []
for img_path, metadata in data.items():
    # Construct natural language description
    description = (
        f"The robot is moving with a speed of {metadata['Body Movement']['vx']}m/s in x-direction, "
        f"{metadata['Body Movement']['vy']}m/s in y-direction, with a rotational speed of {metadata['Body Movement']['v_rot']}rad/s. "
        f"The body position is ({metadata['Body Position']['pos_x']}, {metadata['Body Position']['pos_y']}, {metadata['Body Position']['pos_z']}) meters, "
        f"and the orientation angles are roll: {metadata['Body Rotation']['roll']}°, "
        f"pitch: {metadata['Body Rotation']['pitch']}°, yaw: {metadata['Body Rotation']['yaw']}°."
    )

    # Load image
    image_path = os.path.join("datasets", img_path)  # Ensure path correctness
    image = PILImage.open(image_path).convert("RGB")  # Ensure RGB format

    dataset_list.append({"image": image, "text": description})

# Create Hugging Face dataset
dataset = Dataset.from_list(dataset_list)
dataset = dataset.cast_column("image", Image())  # Convert images to dataset format

# Print example
print(dataset[0])


{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=251x499 at 0x14E55464A60>, 'text': 'The robot is moving with a speed of 0.2m/s in x-direction, -0.04m/s in y-direction, with a rotational speed of 0.32rad/s. The body position is (0.06, -0.09, 0.17) meters, and the orientation angles are roll: -6.2°, pitch: -0.69°, yaw: 165.6°.'}
